# SVM Data Preparation with PCA

here we prepared the data for the SVM training process using PCA dimensionality reduction.



1. Loads the existing train/test split if available.
2. Fits PCA on `x_train` only.
3. Transforms both `x_train` and `x_test`.
4. Saves the PCA-reduced datasets into:

    ```text
    SVM/
    ├── train/
    │   ├── x_train_svm_pca.csv
    │   └── y_train.csv
    └── test/
        ├── x_test_svm_pca.csv
        └── y_test.csv
    ```

PCA is fitted only on the training data to avoid data leakage.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

## Configuration

In [ ]:
CLEANED_DATA_PATH = "cleaned.csv"

BASE_TRAIN_DIR = "train"
BASE_TEST_DIR = "test"

SVM_DIR = "SVM"
SVM_TRAIN_DIR = os.path.join(SVM_DIR, "train")
SVM_TEST_DIR = os.path.join(SVM_DIR, "test")

os.makedirs(SVM_TRAIN_DIR, exist_ok=True)
os.makedirs(SVM_TEST_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.30
PCA_VARIANCE_TO_KEEP = 0.95

## Load the data split

In [ ]:
x_train = pd.read_csv(os.path.join(BASE_TRAIN_DIR, "x_train.csv"))
x_test = pd.read_csv(os.path.join(BASE_TEST_DIR, "x_test.csv"))
y_train = pd.read_csv(os.path.join(BASE_TRAIN_DIR, "y_train.csv")).squeeze("columns")
y_test = pd.read_csv(os.path.join(BASE_TEST_DIR, "y_test.csv")).squeeze("columns")
print("Loaded existing train/test split.")

print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## Apply PCA for SVM

PCA is fitted on `x_train` only, then the same fitted PCA object is used to transform `x_test`.

In [ ]:
pca = PCA(n_components=PCA_VARIANCE_TO_KEEP)

x_train_pca_array = pca.fit_transform(x_train)
x_test_pca_array = pca.transform(x_test)

pc_columns = [f"PC{i+1}" for i in range(x_train_pca_array.shape[1])]

x_train_pca = pd.DataFrame(x_train_pca_array, columns=pc_columns, index=x_train.index)
x_test_pca = pd.DataFrame(x_test_pca_array, columns=pc_columns, index=x_test.index)

print("Original number of features:", x_train.shape[1])
print("Number of PCA components kept:", x_train_pca.shape[1])
print("Total variance preserved:", round(pca.explained_variance_ratio_.sum(), 4))

## Save PCA information

In [ ]:
pca_info = pd.DataFrame({
    "Principal Component": pc_columns,
    "Explained Variance Ratio": pca.explained_variance_ratio_,
    "Cumulative Explained Variance": pca.explained_variance_ratio_.cumsum()
})

pca_info.to_csv(os.path.join(SVM_DIR, "pca_explained_variance.csv"), index=False)

pca_components = pd.DataFrame(
    pca.components_,
    columns=x_train.columns,
    index=pc_columns
)

pca_components.to_csv(os.path.join(SVM_DIR, "pca_components.csv"))

pca_info

## Save SVM-ready datasets

In [ ]:
x_train_pca.to_csv(os.path.join(SVM_TRAIN_DIR, "x_train_svm_pca.csv"), index=False)
x_test_pca.to_csv(os.path.join(SVM_TEST_DIR, "x_test_svm_pca.csv"), index=False)

y_train.to_csv(os.path.join(SVM_TRAIN_DIR, "y_train.csv"), index=False, header=["Churn"])
y_test.to_csv(os.path.join(SVM_TEST_DIR, "y_test.csv"), index=False, header=["Churn"])

print("SVM PCA datasets saved successfully.")
print("Saved files:")
print(os.path.join(SVM_TRAIN_DIR, "x_train_svm_pca.csv"))
print(os.path.join(SVM_TRAIN_DIR, "y_train.csv"))
print(os.path.join(SVM_TEST_DIR, "x_test_svm_pca.csv"))
print(os.path.join(SVM_TEST_DIR, "y_test.csv"))